# Back Translation using MarianMT and Google Translate

This notebook demonstrates how to perform back translation using MarianMT models from Hugging Face and Google Translate. The goal is to augment the dataset by translating text to another language and then back to English.

In [49]:
import argparse
import ast
import os
import random
import sys
import re

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoTokenizer
from transformers.models.marian import MarianMTModel

import nest_asyncio
nest_asyncio.apply()  # For Jupyter; allows nested event loops

import asyncio
from googletrans import Translator, LANGUAGES


In [50]:
class TextDataset(Dataset):
    def __init__(self, tokenizer, original_data_path=None, text_data_list=None):
        self.tokenizer = tokenizer
        if original_data_path:
            self.df = pd.read_csv(original_data_path)
            self.text_data_list = self.df['text'].tolist()
            self.text_num_list = [1] * len(self.text_data_list)
        else:
            self.text_data_list = text_data_list
            self.text_num_list = [1] * len(text_data_list)
    
    def __len__(self):
        return len(self.text_data_list)
    
    def __getitem__(self, idx):
        return self.text_data_list[idx]
    
    def collate_fn(self, batch):
        return self.tokenizer(batch, return_tensors='pt', padding=True, truncation=True)

In [51]:
class BackTranslation:
    def __init__(self, lang="de"):
        self.lang = lang
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        try:
            print("USING:", self.device)
            self.en_lang_tokenizer = AutoTokenizer.from_pretrained(f"Helsinki-NLP/opus-mt-en-{lang}")
            self.lang_en_tokenizer = AutoTokenizer.from_pretrained(f"Helsinki-NLP/opus-mt-{lang}-en")
            self.en_lang_translator = MarianMTModel.from_pretrained(f"Helsinki-NLP/opus-mt-en-{lang}").to(self.device)
            self.lang_en_translator = MarianMTModel.from_pretrained(f"Helsinki-NLP/opus-mt-{lang}-en").to(self.device)
        except Exception as e:
            print(f"Warning: Could not load models for language {lang}: {str(e)}")
            self.en_lang_tokenizer = None

    def do_back_translation(self, original_data_path, batch_size, temperature, **generate_kwargs):
        if not self.en_lang_tokenizer:  # Skip if model loading failed
            return None, None
            
        assert len(temperature) <= 2
        temp1, temp2 = (temperature[0], temperature[0]) if len(temperature) == 1 else temperature
        pandas_text_dataset = TextDataset(self.en_lang_tokenizer, original_data_path=original_data_path)
        dataloader = DataLoader(
            pandas_text_dataset, shuffle=False, drop_last=False, num_workers=4, 
            batch_size=batch_size, collate_fn=pandas_text_dataset.collate_fn
        )
        text_num_list = pandas_text_dataset.text_num_list
        lang_out_list = []
        for batch in tqdm(dataloader, desc=f"Translating to {self.lang}"):
            lang_out = self.en_lang_translator.generate(**batch.to(self.device), temperature=temp1, **generate_kwargs)
            for out in lang_out:
                lang_out_list.append(self.en_lang_tokenizer.decode(out).replace("<pad>", "").replace("</s>", "").strip())
        
        lang_text_dataset = TextDataset(self.lang_en_tokenizer, text_data_list=lang_out_list)
        dataloader = DataLoader(
            lang_text_dataset, shuffle=False, drop_last=False, num_workers=4, 
            batch_size=batch_size, collate_fn=lang_text_dataset.collate_fn
        )
        en_out_list = []
        for batch in tqdm(dataloader, desc=f"Translating back from {self.lang}"):
            en_out = self.lang_en_translator.generate(**batch.to(self.device), temperature=temp2, **generate_kwargs)
            for out in en_out:
                en_out_list.append(self.lang_en_tokenizer.decode(out).replace("<pad>", "").replace("</s>", "").strip())
        
        text_augment_list = []
        start = 0
        for text_num in text_num_list:
            text_augment_list.append(en_out_list[start : start + text_num])
            start += text_num
            
        return lang_out_list, text_augment_list

In [52]:
def test_backtranslation():
    sample_data = pd.DataFrame({
        'text': ["The cat is on the mat", "Dogs love to play fetch"]
    })
    os.makedirs("dataset", exist_ok=True)
    sample_data.to_csv("dataset/dataset_train.csv", index=False)
    bt = BackTranslation(lang="de")
    bt.do_back_translation(
        original_data_path="dataset/dataset_train.csv",
        batch_size=256,
        temperature=[1.0],
        num_beams=5,
        do_sample=True
    )
    result_df = pd.read_csv("dataset/dataset_aug_train.csv")
    print("\nOriginal and Augmented Texts:")
    print(result_df[['text', 'text_augment']])
    with open("dataset/dataset_aug_train.de.txt", "r") as f:
        print("\nIntermediate German Translation:")
        print(f.read())

In [53]:
# import sys

# if __name__ == "__main__":
#     parser = argparse.ArgumentParser()
#     parser.add_argument("--lang", type=str, default="de")
#     parser.add_argument("--temperature", nargs="+", type=float, default=[1.0])
#     parser.add_argument("--seed", type=int, default=42)
#     parser.add_argument("--bsz", type=int, default=128)
#     parser.add_argument("--num-beams", type=int, default=5)
    
#     # Parse args and ignore unknown ones (like Jupyter's -f argument)
#     args, unknown = parser.parse_known_args()
    
#     # Set random seed if needed
#     torch.manual_seed(args.seed)
    
#     # Run the test
#     test_backtranslation()
    
#     # Uncomment this section to run the original main loop instead
#     """
#     backtranslation = BackTranslation(lang=args.lang)
#     for data_split in ["train", "valid"]:
#         inp_data_path = f"dataset/dataset_{data_split}.csv"
#         out_data_path = f"dataset/dataset_aug_{data_split}.csv"
#         if not os.path.exists(os.path.dirname(out_data_path)):
#             os.makedirs(os.path.dirname(out_data_path))
#         backtranslation.do_back_translation(
#             inp_data_path, 
#             out_data_path, 
#             batch_size=args.bsz, 
#             num_beams=args.num_beams, 
#             temperature=args.temperature, 
#             do_sample=True
#         )
#     """

    

In [54]:
async def googletrans_back_translate_with_suffix(df, lang_list, suffix="_gcp", text_col="text"):
    translator = Translator()

    for lang in lang_list:
        
        intermediate_col = f"intermediate_{lang}{suffix}"
        augment_col = f"augment_{lang}{suffix}"
        
        df[intermediate_col] = pd.Series(dtype="string")
        df[augment_col] = pd.Series(dtype="string")
        
        # Process each row for this language.
        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"GoogleTrans Translating {lang}"):
            original_text = row.get(text_col, "")
            if not isinstance(original_text, str) or not original_text.strip():
                continue  # Skip rows without valid text
            
            try:
                # Forward translation: English -> target language.
                translation = await translator.translate(original_text, dest=lang)
                intermediate_text = translation.text if translation.text else original_text
            except Exception as e:
                print(f"[Error] GoogleTrans translating row {idx} to {lang}: {e}")
                intermediate_text = original_text
            
            try:
                # Back translation: target language -> English.
                back_translation = await translator.translate(intermediate_text, dest="en")
                augment_text = back_translation.text if back_translation.text else original_text
            except Exception as e:
                print(f"[Error] GoogleTrans back translating row {idx} from {lang}: {e}")
                augment_text = original_text
            
            df.at[idx, intermediate_col] = intermediate_text
            df.at[idx, augment_col] = augment_text

    return df

In [58]:
async def run_multi_language_pipeline():
    languages_set1 = ['af', 'sq', 'ar', 'hy', 'eu', 'bg', 'ca', 'zh', 'cs', 'da', 'nl', 'et', 'fi', 'fr', 'gl', 'de', 'ht', 'hi', 'hu', 'is', 'id', 'ga', 'it', 'mk', 'ml', 'mt', 'mr', 'ru', 
                      'sk', 'es', 'sv', 'uk', 'ur', 'vi', 'cy'] 

    languages_set2 = ['bn', 'hr', 'ka', 'el', 'gu', 'he', 'ja', 'kn', 'kk', 'km', 'ko', 'lv', 'lt', 'ms', 'ne', 'no', 'fa', 'pl', 'pt', 'pa', 'ro', 'sr', 'sl', 'sw', 'ta', 'te', 'th', 'tr', 
                      'yi', 'zu']

    # Input and output paths
    original_data_path = "dataset_Small/dataset_train.csv"
    output_path = "dataset_Small/dataset_aug_train_all_new.csv"
    print("FLAG")
    # Parameters
    batch_size = 128
    temperature = [1.0]
    num_beams = 5
    

    # Load the original dataset
    original_df = pd.read_csv(original_data_path)
    print(len(original_df))

    # Process each language_set1 using Helsinki and store results (with _hels to distinguish)
    for lang in languages_set1:
        print(f"\nProcessing language: {lang}")
        
        bt = BackTranslation(lang=lang)
        intermediate_texts, augmented_texts = bt.do_back_translation(
            original_data_path=original_data_path,
            batch_size=batch_size,
            temperature=temperature,
            num_beams=num_beams,
            do_sample=True
        )
        
        if intermediate_texts and augmented_texts:
            # Add columns for this language
            original_df[f'intermediate_{lang}_hels'] = intermediate_texts
            original_df[f'augment_{lang}_hels'] = augmented_texts
        else:
            print(f"Skipping {lang} due to model loading failure")
            original_df[f'intermediate_{lang}_hels'] = None
            original_df[f'augment_{lang}_hels'] = None


    # Process languages_set2 using Google Translate and add results to original_df (with _gcp to distinguish)
    original_df = await googletrans_back_translate_with_suffix(original_df, lang_list=languages_set2, suffix="_gcp", text_col="text")
        

    # Save all results to a single CSV
    original_df.to_csv(output_path, index=False)
    print(f"\nResults saved to {output_path}")
    
    # Print the results
    print("\nFinal Results:")
    print(original_df)

In [59]:
if __name__ == "__main__":
    # Set random seed
    torch.manual_seed(42)
    await run_multi_language_pipeline()  #jupyter specific

FLAG
2

Processing language: af
USING: cuda


Translating back from af: 100%|██████████| 1/1 [00:00<00:00,  4.79it/s]



Processing language: sq
USING: cuda


Translating back from sq: 100%|██████████| 1/1 [00:00<00:00,  4.18it/s]



Processing language: ar
USING: cuda


...


Processing language: vi
USING: cuda


Translating back from vi: 100%|██████████| 1/1 [00:00<00:00,  3.42it/s]



Processing language: cy
USING: cuda


GoogleTrans Translating zu: 100%|██████████| 2/2 [00:00<00:00, 10.73it/s]



Results saved to dataset_Small/dataset_aug_train_all_new.csv

Final Results:
                      text               intermediate_af_hels  \
0    The cat is on the mat              Die kat is op die mat   
1  Dogs love to play fetch  Honde hou daarvan om te gaan haal   

           augment_af_hels            intermediate_sq_hels  \
0  [The cat's on the mat.]         Macja është në rrogozë.   
1       [Dogs Love to Get]  Qenve u pëlqen shumë të luajnë   

           augment_sq_hels intermediate_ar_hels          augment_ar_hels  \
0  [The cat's on the mat.]   القطة على ما سُجّل  [The cat is on record.]   
1     [Dogs love to play.]   الكلاب تحب أن تلعب     [Dogs like to play.]   

             intermediate_hy_hels                   augment_hy_hels  \
0  Ցայտունացումը մաշմալլոնի վրա է  [The Tropics are on the Tropics]   
1        Շունը սիրում է խաղ խաղալ         [The game loves playing.]   

           intermediate_eu_hels  ...          intermediate_te_gcp  \
0         Katua alfonbran d

In [ ]:
# If running as a standalone script, use the following:
# if __name__ == "__main__":
#     asyncio.run(run_multi_language_pipeline())

In [60]:
# Code to clean the csv (remove brackets and quotes if they are not part of the original text) and save it to a new file

def clean_augmented_text(aug_text, orig_text):
    """
    Remove 
    """
    # If aug_text is not a string, return as-is.
    if not isinstance(aug_text, str):
        return aug_text

    cleaned = aug_text.strip()
    
    # Remove outer brackets if they exist.
    if cleaned.startswith('[') and cleaned.endswith(']'):
        cleaned = cleaned[1:-1].strip()
    
    # Remove outer quotes if they exist.
    if (cleaned.startswith('"') and cleaned.endswith('"')) or \
       (cleaned.startswith("'") and cleaned.endswith("'")):
        quote_char = cleaned[0]
        
        # Only remove the quotes if the original text doesn't also start and end with the same quote.
        if not (isinstance(orig_text, str) and 
                orig_text.strip().startswith(quote_char) and 
                orig_text.strip().endswith(quote_char)):
            cleaned = cleaned[1:-1].strip()
    
    return cleaned


df = pd.read_csv('dataset_Small/dataset_aug_train_all_new.csv')

# For every column that starts with "augment_", clean the text.
for col in df.columns:
    if col.startswith("augment_"):
        df[col] = df.apply(lambda row: clean_augmented_text(row[col], row["text"]), axis=1)

# Save the cleaned DataFrame to a new CSV.
output_csv = 'dataset_Small/dataset_aug_train_all_new_clean.csv'
df.to_csv(output_csv, index=False)
print(f"Cleaned CSV saved to: {output_csv}")


Cleaned CSV saved to: dataset_Small/dataset_aug_train_all_new_clean.csv


In [ ]:
# Script to verify if there's anything missing - output you see saved below is just an example from a prev test, nothing's missing now

def is_missing(val):
    """Return True if val is NaN or a string that is empty (after stripping whitespace)."""
    if pd.isna(val):
        return True
    if isinstance(val, str) and val.strip() == "":
        return True
    return False

def check_missing_translations(csv_file):
    df = pd.read_csv(csv_file)
    missing_info = {}
    
    for col in df.columns:
        if col.startswith("intermediate_") or col.startswith("augment_"):
            missing_count = df[col].apply(is_missing).sum()

            # Record if there are missing values.
            if missing_count > 0:

                # For columns like "intermediate_hr_hels" or "augment_hr_gcp", assume the language code is the second token when splitting by underscore.
                tokens = col.split("_")
                if len(tokens) < 2:
                    continue
                lang = tokens[1]
                
                # Determine if this column is for forward translation or back translation.
                key = "translation" if col.startswith("intermediate_") else "backtranslation"
                
                if lang not in missing_info:
                    missing_info[lang] = {"translation": 0, "backtranslation": 0}
                missing_info[lang][key] += missing_count
    
    return missing_info

# CSV path
csv_file = 'dataset_Small/dataset_aug_train_all_new_clean.csv'
missing_info = check_missing_translations(csv_file)

if not missing_info:
    print("No missing translation or backtranslation entries found.")
else:
    print("Missing translation/backtranslation info:")
    for lang, info in missing_info.items():
        print(f"Language '{lang}': {info}")

Missing translation/backtranslation info:
Language 'bn': {'translation': np.int64(2), 'backtranslation': np.int64(2)}


In [ ]:
# Run this only if you want to check if the lang code is available in Helsinki. Some two letter lang codes have been replaced three letter codes in Helsinki. 
# This was just for 2-3 languages like ja and sl, so I'm sending those to google anyway.

import requests
import time

def model_exists(lang):
    try:
        device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

        en_lang_tokenizer = AutoTokenizer.from_pretrained(f"Helsinki-NLP/opus-mt-en-{lang}")
        lang_en_tokenizer = AutoTokenizer.from_pretrained(f"Helsinki-NLP/opus-mt-{lang}-en")
        en_lang_translator = MarianMTModel.from_pretrained(f"Helsinki-NLP/opus-mt-en-{lang}").to(device)
        lang_en_translator = MarianMTModel.from_pretrained(f"Helsinki-NLP/opus-mt-{lang}-en").to(device)
        return True
    
    except Exception as e:
        print(f"Warning: Could not load models for language {lang}: {str(e)}")
        return False

languages = ["af", "sq", "ar", "hy", "eu", "bg", "bn", "ca", "zh", "hr", "cs", "da", "nl", "et", "fi", "fr", "gl", "ka", "de", "el", "gu", "ht", "he", "hi", "hu", "is", "id", "ga", "it", "ja", 
             "kn", "kk", "km", "ko", "lv", "lt", "mk", "ms", "ml", "mt", "mr", "ne", "no", "fa", "pl", "pt", "pa", "ro", "ru", "sr", "sk", "sl", "es", "sw", "sv", "ta", "te", "th", "tr", "uk", 
             "ur", "vi", "cy", "yi", "zu"]   

available = []
missing = []

for lang in languages:
    time.sleep(0.2)
    if model_exists(lang):
        available.append(lang)
    else:
        missing.append(lang)

print("\nLanguages with available models:", len(available), available)
print("Languages with missing models:", len(missing), missing)

If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingf

pytorch_model.bin:  29%|##8       | 83.9M/294M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/294M [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/813k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/290M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/290M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

Languages with available models: 35 ['af', 'sq', 'ar', 'hy', 'eu', 'bg', 'ca', 'zh', 'cs', 'da', 'nl', 'et', 'fi', 'fr', 'gl', 'de', 'ht', 'hi', 'hu', 'is', 'id', 'ga', 'it', 'mk', 'ml', 'mt', 'mr', 'ru', 'sk', 'es', 'sv', 'uk', 'ur', 'vi', 'cy']
Languages with missing models: 30 ['bn', 'hr', 'ka', 'el', 'gu', 'he', 'ja', 'kn', 'kk', 'km', 'ko', 'lv', 'lt', 'ms', 'ne', 'no', 'fa', 'pl', 'pt', 'pa', 'ro', 'sr', 'sl', 'sw', 'ta', 'te', 'th', 'tr', 'yi', 'zu']
